# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the provided URL and covers ordered logistic regression outputs for pastoralist households in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review all available record sets including their `@id`s and the fields (`@id`s and names) contained in each.

In [ ]:
# List all record sets by @id and their fields
print("Record sets available in this dataset:")
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets were found in the Croissant metadata. Note: some datasets provide only documentation-related metadata or require access permissions for data files.")
else:
    for rs in record_sets:
        print(f"\nRecord Set: {rs['@id']}\n  Name: {rs.get('name', '<no name>')}\n  Fields:")
        for field in rs.get('fields', []):
            print(f"    - @id: {field['@id']:40} | name: {field.get('name', '<no name>')}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Each record set is referenced by its `@id` as shown above.

***Note:*** For demonstration purposes, below we attempt to extract records from all available record sets. If there are no downloadable data record sets, code will show this and skip extraction.

In [ ]:
# Prepare to collect records from all discovered record sets
dataframes = {}
all_recordset_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []

if not all_recordset_ids:
    print("No record sets to extract records from.")
else:
    for record_set_id in all_recordset_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded {len(df)} records from record set {record_set_id}.")
            else:
                print(f"No records found in record set {record_set_id}.")
        except Exception as e:
            print(f"Could not load records from {record_set_id}: {str(e)}")
    if dataframes:
        # Pick first record set with data for demonstration
        first_rs_id = next(iter(dataframes))
        print(f"\nColumns in first populated record set ({first_rs_id}):")
        print(dataframes[first_rs_id].columns.tolist())
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply EDA steps such as filtering, normalization, and grouping using one of the numeric fields from a data-containing record set.

***Note:*** Update the code below after inspecting previous cell's output to select meaningful field `@id`s corresponding to numeric and grouping variables. If no data is available, EDA steps will be skipped.

In [ ]:
import numpy as np

if not dataframes:
    print("No dataframes available for EDA. Please check if the record set(s) contain downloadable records.")
else:
    # Pick the first available dataframe and its record set @id
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    # Attempt to find numeric columns for analysis (float or int columns)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        print(f"No numeric columns found in record set {record_set_id} for EDA.")
    else:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()
        # Filter records above the average for the chosen field
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in {record_set_id} with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize selected field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a non-numeric column, if present
        non_numeric_cols = [col for col in df.columns if col not in numeric_cols]
        group_field = non_numeric_cols[0] if non_numeric_cols else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No non-numeric grouping field found.")

## 5. Visualization
Visualize data distributions or relationships among fields. Here, we provide an example visualization using available numeric data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data available for visualization.")
else:
    # Use the same numeric_field and record set as above
    df = dataframes[record_set_id]
    if numeric_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field} in {record_set_id}')
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        if len(numeric_cols) > 1:
            y_field = numeric_cols[1]
            plt.figure(figsize=(6,6))
            sns.scatterplot(x=df[numeric_field], y=df[y_field])
            plt.title(f'{numeric_field} vs {y_field}')
            plt.xlabel(numeric_field)
            plt.ylabel(y_field)
            plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library. We performed step-by-step metadata inspection, tabular record extraction, simple EDA, and visualization using dataset entity `@id`s to ensure reproducibility and clarity.

For deeper domain insights, further feature engineering and statistical modeling can be conducted using the provided DataFrames. Consult the dataset documentation (metadata fields) for ethical and usage considerations.